In [1]:
import pandas as pd
import os

def load_csv(file_path, vlm_name):
    df = pd.read_csv(file_path)
    if 'file_path' in df.columns:
        df['image_name'] = df['file_path'].apply(os.path.basename)
        df = df[['image_name', 'file_path', 'vlm_output']].rename(
            columns={'file_path': 'path', 'vlm_output': vlm_name}
        )
    else:
        df = df[['image_name', 'path', 'vlm_answer']].rename(
            columns={'vlm_answer': vlm_name}
        )
    return df

def merge_all():
    internvl = load_csv("/home/kirtangangani/satellite_data/VLM/intern_vl_0_hr_updated.csv", "InternVL")
    # paligemma = load_csv("/home/kirtangangani/satellite_data/VLM/paligemma_0_hr.csv", "PaliGemma")
    qwen = load_csv("/home/kirtangangani/satellite_data/VLM/qwen_0_hr_updated.csv", "Qwen2VL")
    geochat = load_csv("/home/devansh.lodha/GeoChat/geochat_captions_final.csv", "GeoChat7B")

    df = internvl.merge(qwen, on=['image_name', 'path'], how='outer') \
                 .merge(geochat, on=['image_name', 'path'], how='outer')

    df.to_csv("combined_vlm_outputs_grouped.csv", index=False)
    print("Saved as combined_vlm_outputs_grouped.csv")
    print(df.head())

if __name__ == "__main__":
    merge_all()

Saved as combined_vlm_outputs_grouped.csv
           image_name                                               path  \
0     2485_1627_0.png  /home/rishabh.mondal/Brick-Kilns-project/ijcai...   
1  2507_1627_2100.png  /home/rishabh.mondal/Brick-Kilns-project/ijcai...   
2    769_1400_700.png  /home/rishabh.mondal/Brick-Kilns-project/ijcai...   
3     538_700_700.png  /home/rishabh.mondal/Brick-Kilns-project/ijcai...   
4   2541_1400_700.png  /home/rishabh.mondal/Brick-Kilns-project/ijcai...   

                                            InternVL  \
0                                A dark, cloudy sky.   
1  Aerial view of a foggy industrial area with bu...   
2  Aerial view of a dry riverbed with intricate e...   
3    Aerial view of a forest fire with smoke rising.   
4  Aerial view of a coastal area with boats in th...   

                                             Qwen2VL  \
0                             "Whispers in the Dark"   
1  "An aerial view of industrial buildings shroud...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Wrote combined_vlm_outputs_with_gptoss_caption_objects.csv
     image_name                                                                                                                                                  path summary_gptoss_caption summary_gptoss_objects
2485_1627_0.png /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/xview/processed/split/train/images/2485_1627_0.png  final concise caption  ..., object1, object2


In [4]:
import pandas as pd
import re
import torch
from transformers import pipeline

# Input/Output
INPUT_CSV = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/combined_vlm_outputs_grouped.csv"
OUTPUT_CSV = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/combined_vlm_with_oss_summary.csv"

# Model
MODEL_ID = "openai/gpt-oss-20b"

# Caption columns present in the CSV
CAPTION_COLS = ["InternVL", "PaliGemma", "Qwen2VL", "GeoChat7B"]

# Minimal cleaner
def clean(txt: str) -> str:
    if not isinstance(txt, str):
        return ""
    t = txt.strip().replace('"""', '"')
    t = re.sub(r"\s+", " ", t)
    return t.strip().strip('"').strip("'")

# Prompt builder: short, to the point
def build_prompt(image_name: str, path: str, caps: list[str]) -> str:
    caps = [f"- {c}" for c in caps if c]
    caps_block = "\n".join(caps)
    return (
        "Summarize these 4 captions of the SAME satellite image into ONE concise caption.\n"
        "Rules: keep only scene-relevant info; remove poetic/subjective words; merge duplicates; "
        "prefer standard land-cover terms (buildings, roads, water, vegetation, barren land, cloud cover, shadows); "
        "mention rough location cues if present (top-left, bottom-right, center). "
        "Output: one sentence under 40 words.\n\n"
        f"Image: {image_name}\nPath: {path}\nCaptions:\n{caps_block}\n\nFinal:"
    )

def main():
    # Load data
    df = pd.read_csv(INPUT_CSV)

    # HF pipeline
    pipe = pipeline(
        "text-generation",
        model=MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )

    summaries = []
    for _, row in df.iterrows():
        caps = [clean(row.get(c, "")) for c in CAPTION_COLS]
        caps = [c for c in caps if c]
        prompt = build_prompt(str(row.get("image_name","")), str(row.get("path","")), caps)

        # Generate
        out = pipe(prompt, max_new_tokens=64, temperature=0.2, top_p=0.9, do_sample=False)
        text = out[0]["generated_text"]

        # Heuristic: take only the part after "Final:" if model echoed prompt
        if "Final:" in text:
            text = text.split("Final:", 1)[-1].strip()

        # Post-trim to ~60 words
        words = text.split()
        if len(words) > 60:
            text = " ".join(words[:60])

        summaries.append(text)

    df["summary_gptoss"] = summaries
    df.to_csv(OUTPUT_CSV, index=False)

if __name__ == "__main__":
    main()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


KeyboardInterrupt: 

In [1]:
import pandas as pd
import re
import torch
from transformers import pipeline

# Input/Output
INPUT_CSV = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/combined_vlm_outputs_grouped.csv"
OUTPUT_CSV = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/cvpr/combined_vlm_with_oss_summary.csv"

# Model
MODEL_ID = "openai/gpt-oss-20b"

# Caption columns present in the CSV
CAPTION_COLS = ["InternVL", "PaliGemma", "Qwen2VL", "GeoChat7B"]

# Minimal cleaner
def clean(txt: str) -> str:
    if not isinstance(txt, str):
        return ""
    t = txt.strip().replace('"""', '"')
    t = re.sub(r"\s+", " ", t)
    return t.strip().strip('"').strip("'")

# Prompt builder
def build_prompt(image_name: str, path: str, caps: list[str]) -> str:
    caps = [f"- {c}" for c in caps if c]
    caps_block = "\n".join(caps)
    return (
        "Summarize these 4 captions of the SAME satellite image into ONE concise caption.\n"
        "Rules: keep only scene-relevant info; remove poetic or subjective words; merge duplicates; "
        "prefer standard land-cover and object terms (buildings, roads, water, vegetation, barren land, "
        "cloud cover, shadows, vehicles, industrial sites, bridges, rivers, mountains, etc.); "
        "mention rough location cues if present (top-left, bottom-right, center); "
        "add object names that describe visible structures or terrain. "
        "Output one sentence under 60 words summarizing the image scene.\n\n"
        f"Image: {image_name}\nPath: {path}\nCaptions:\n{caps_block}\n\nFinal:"
    )

def main():
    # Load data
    df = pd.read_csv(INPUT_CSV)

    # HF pipeline
    pipe = pipeline(
        "text-generation",
        model=MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )

    summaries = []
    total = len(df)

    for i, row in df.iterrows():
        caps = [clean(row.get(c, "")) for c in CAPTION_COLS]
        caps = [c for c in caps if c]
        prompt = build_prompt(str(row.get("image_name", "")), str(row.get("path", "")), caps)

        # Generate
        out = pipe(prompt, max_new_tokens=64, temperature=0.3, top_p=0.9, do_sample=True)
        text = out[0]["generated_text"]

        # Heuristic: extract part after "Final:"
        if "Final:" in text:
            text = text.split("Final:", 1)[-1].strip()

        # Trim to ~60 words
        words = text.split()
        if len(words) > 60:
            text = " ".join(words[:60])

        summaries.append(text)

        # Progress tracker every 2 rows
        if (i + 1) % 1 == 0:
            print(f"[Progress] Completed {i + 1} / {total} images")

    df["summary_gptoss"] = summaries
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"[Done] Saved summaries to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2025-11-13 01:32:18.849498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762977738.870387    1371 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762977738.878465    1371 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762977738.906288    1371 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762977738.906309    1371 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762977738.906311    1371 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


[Progress] Completed 1 / 40283 images
[Progress] Completed 2 / 40283 images
[Progress] Completed 3 / 40283 images
[Progress] Completed 4 / 40283 images
[Progress] Completed 5 / 40283 images
[Progress] Completed 6 / 40283 images
[Progress] Completed 7 / 40283 images
[Progress] Completed 8 / 40283 images
[Progress] Completed 9 / 40283 images


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[Progress] Completed 10 / 40283 images
[Progress] Completed 11 / 40283 images
[Progress] Completed 12 / 40283 images
[Progress] Completed 13 / 40283 images
[Progress] Completed 14 / 40283 images
[Progress] Completed 15 / 40283 images
[Progress] Completed 16 / 40283 images
[Progress] Completed 17 / 40283 images
[Progress] Completed 18 / 40283 images
[Progress] Completed 19 / 40283 images
[Progress] Completed 20 / 40283 images
[Progress] Completed 21 / 40283 images
[Progress] Completed 22 / 40283 images
[Progress] Completed 23 / 40283 images
[Progress] Completed 24 / 40283 images
[Progress] Completed 25 / 40283 images
[Progress] Completed 26 / 40283 images
[Progress] Completed 27 / 40283 images
[Progress] Completed 28 / 40283 images
[Progress] Completed 29 / 40283 images
[Progress] Completed 30 / 40283 images
[Progress] Completed 31 / 40283 images
[Progress] Completed 32 / 40283 images
[Progress] Completed 33 / 40283 images
[Progress] Completed 34 / 40283 images
[Progress] Completed 35 /

KeyboardInterrupt: 